# GEP analysis template

Step-by-step version of `bin/gep_analysis.py` (the command line / `--step analysis` Nextflow step), using the same
functions from `bin/gep_lib`. Run it from the `cnmf` project directory in the `envs/cnmf-analysis.sif` environment
(scanpy, decoupler, seaborn, openpyxl). Edit the parameters in the next cell, then run the cells in order; each step can be
rerun or changed on its own.

In [ ]:
import sys
sys.path.insert(0, 'bin')          # so `gep_lib` can be imported from the project directory

from pathlib import Path
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import Image, display

import gep_lib as gl

gl.set_publication_style(dpi=300)

## Parameters

In [ ]:
sample = 'ctype_MEL'                                    # sample folder (the row tag in the samplesheet)
outdir = Path('results_test')                           # the pipeline's results folder
run_name = Path(outdir / sample / 'pipeline_info' / 'run_name.txt').read_text().strip()
k, threshold = 3, 2                                     # the values given to the consensus step

adata_file = outdir / sample / 'anndatas' / f'adata_topometry_{sample}.h5ad'
run_dir = outdir / sample / run_name                    # cNMF results folder
out = outdir / sample / 'analysis'                      # outputs (csvs/, excel/, figures/, anndatas/) go here

gmt = 'assets/resources/MOUSE_GO_bp_no_GO_iea_symbol.gmt'   # from bin/fetch_resources.py
collectri = 'assets/resources/collectri_mouse.parquet'

projection = 'projection'      # obsm key of the embedding to plot on
cluster_col, age_col = 'clusters', 'age_pretty'
summary_cols = ['age_pretty', 'clusters', 'ctype_detailed']   # missing columns are skipped
usage_cutoff = 0.1             # usage above which a cell counts as using a GEP
formats = ['png']              # add 'pdf' for vector text

figs = out / 'figures'
for d in ('csvs', 'excel', 'figures', 'anndatas'):
    (out / d).mkdir(parents=True, exist_ok=True)

## Load the AnnData and merge in the cNMF results

In [ ]:
adata = ad.read_h5ad(adata_file)
usage, scores = gl.load_gep_results(run_dir, run_name, k, threshold)   # usage: cells x GEPs, scores: genes x GEPs
gl.merge_gep_results(adata, usage, scores, run_name, k, threshold)
geps = [gl.gep_number(c) for c in gl.gep_columns(adata)]
adata

## Top genes per GEP

In [ ]:
top = gl.top_genes(scores, n=100)
top.to_csv(out / 'csvs' / 'top100_genes.csv', index=False)
gl.write_table_workbook(top, out / 'excel' / 'top100_genes.xlsx')
top.head(10)

## Enrichment (GO biological process and CollecTRI)
Each GEP's gene scores are tested against every gene set at once with decoupler's univariate linear model. Only gene sets
with p < 0.05 are kept.

In [ ]:
results = {}
for name, net in (('gobp', gl.load_gobp(gmt)), ('collectri', gl.load_collectri(collectri))):
    results[name] = gl.run_ulm(scores, net, pval=0.05)
    results[name].to_csv(out / 'csvs' / f'gsea_{name}.csv', index=False)
    gl.write_gep_workbook(results[name], out / 'excel' / f'gsea_{name}.xlsx',
                          {'method': 'decoupler ULM', 'name': name}, n_geps=len(geps))
results['gobp'].head()

## Expression of the top genes
scanpy's standard `normalize_total` + `log1p` (no scaling), from the raw counts of all genes, for the plotted genes only.

In [ ]:
genes = list(dict.fromkeys(g for gep in top.columns for g in top[gep].iloc[:3]))
total = np.asarray(adata.raw.X.sum(axis=1)).ravel()                    # each cell's counts over all genes
counts = adata.raw[:, genes].X
counts = counts.toarray() if sp.issparse(counts) else counts
expr = ad.AnnData(np.log1p(counts / total[:, None] * 1e4), obs=adata.obs,
                  var=pd.DataFrame(index=genes), obsm={projection: adata.obsm[projection]})

## Usage summaries

In [ ]:
palettes = gl.group_palettes(adata, age_col, cluster_col, [c for c in summary_cols if c not in (age_col, cluster_col)])

paths, pct = gl.plot_usage_by_cluster(adata, cluster_col, palettes, usage_cutoff, figs / 'gep_usage_pct_by_cluster', formats)
display(Image(str(paths[0])))

In [ ]:
paths, pct = gl.plot_usage_by_age(adata, age_col, palettes, usage_cutoff, figs / 'gep_usage_pct_by_age', formats)
display(Image(str(paths[0])))

In [ ]:
paths, pct = gl.plot_usage_by_cluster_and_age(adata, cluster_col, age_col, palettes, usage_cutoff,
                                              figs / 'gep_usage_heatmap_cluster_age', formats, min_cells=10)
display(Image(str(paths[0])))

## Projection summary

In [ ]:
paths = gl.plot_projection_summary(adata, projection, summary_cols, palettes, cluster_col, figs / 'projection_summary', formats)
display(Image(str(paths[0])))

## Per-GEP figures
`gl.MEGAPLOT` is the full row (usage, top 3 genes, GO, CollecTRI); `gl.USAGE_GENES` is the first four panels. Try a
different set of panels by passing your own tuple, e.g. `('usage', 'gobp', 'collectri')`.

In [ ]:
page = geps[:4]                                       # a few GEPs; loop over gl.page_ranges(geps, 6) for all of them
paths = gl.plot_gep_grid(adata, expr, page, gl.MEGAPLOT, top, results, projection,
                         figs / f'gep_megaplot_{gl.page_label(page)}', formats)
display(Image(str(paths[0])))

In [ ]:
# one panel of one GEP on its own, e.g. GEP 1's GO result
paths = gl.plot_gep_element(adata, expr, 1, 'gobp', top, results, projection, figs / 'elements' / 'gep_01_gobp', formats)
display(Image(str(paths[0])))

## Save the merged AnnData

In [ ]:
adata.write_h5ad(out / 'anndatas' / f'adata_cnmf_{sample}.h5ad')